# Genie Views Setup — Flatten MMF Array Outputs for Natural-Language Access

`mmf.fresh_retail_net.scoring_output` and `evaluation_output` store forecasts as **ARRAY columns** (7-element, one per horizon day) plus a **binary `model_pickle`**. Databricks Genie's NL→SQL works on flat scalar columns; it cannot reason over array/binary columns. This notebook creates **exploded views** so any Databricks-on-AWS user can put a Genie Space (or any BI/agent consumer, e.g. Amazon Quick) over MMF output.


## 1. Configuration

In [ ]:
CATALOG = "mmf"
SCHEMA  = "fresh_retail_net"
USE_MATERIALIZED = True   # True -> MATERIALIZED VIEW (needs serverless); False -> plain VIEW

# MMF selects the best model per series across many models — the default here exposes ALL models
# so Genie can answer model-comparison questions ("which model is most accurate?"), which is core to MMF.
# Optionally set to a single model name (e.g. "Chronos2") to scope a focused single-model demo.
MODEL_FILTER = None   # None = all models (recommended); or a model name (e.g. "Chronos2") for a focused demo

src = lambda t: f"{CATALOG}.{SCHEMA}.{t}"
# WHERE fragment applied inside each view (aliasing-aware variants built per cell).
print("MODEL_FILTER:", MODEL_FILTER)
print("Source tables:")
for t in ["scoring_output", "evaluation_output"]:
    print(f"  {src(t)}: {spark.table(src(t)).count():,} rows")

## 2. Helper: create as MV, fall back to VIEW on failure

In [ ]:
def create_view(name, select_sql):
    """Create name as a MATERIALIZED VIEW if USE_MATERIALIZED, else a VIEW.
    Falls back to plain VIEW if MV creation is unsupported on this compute."""
    fq = f"{CATALOG}.{SCHEMA}.{name}"
    if USE_MATERIALIZED:
        try:
            spark.sql(f"DROP MATERIALIZED VIEW IF EXISTS {fq}")
            spark.sql(f"CREATE MATERIALIZED VIEW {fq} AS\n{select_sql}")
            print(f"  created MATERIALIZED VIEW {fq}")
            return
        except Exception as e:
            print(f"  MV failed for {fq} ({type(e).__name__}); falling back to plain VIEW.")
    spark.sql(f"DROP VIEW IF EXISTS {fq}")
    spark.sql(f"CREATE VIEW {fq} AS\n{select_sql}")
    print(f"  created VIEW {fq}")

## 3. `scoring_output_mv` — exploded forecasts

`ds` and `y` are aligned 7-element arrays. `arrays_zip` + `explode` gives one row per forecast day.
The binary `model_pickle` and `model_uri` are dropped.

In [ ]:
_where_s = f"WHERE s.model = '{MODEL_FILTER}'" if MODEL_FILTER else ""
create_view("scoring_output_mv", f"""
SELECT
  s.unique_id,
  z.ds AS forecast_date,
  z.y  AS forecast_value,
  s.model,
  s.run_id,
  s.run_date,
  s.use_case
FROM {src('scoring_output')} s
LATERAL VIEW explode(arrays_zip(s.ds, s.y)) t AS z
{_where_s}
""")

## 4. `evaluation_metrics_mv` — scalar backtest metrics

Keeps only scalar columns (per-series SMAPE etc.); drops the forecast/actual arrays and binary pickle.
This is what answers “which model is most accurate?” in Genie.

In [ ]:
_where_m = f"WHERE model = '{MODEL_FILTER}'" if MODEL_FILTER else ""
create_view("evaluation_metrics_mv", f"""
SELECT
  unique_id,
  model,
  metric_name,
  metric_value,
  backtest_window_start_date,
  run_id,
  run_date,
  use_case
FROM {src('evaluation_output')}
{_where_m}
""")

## 5. `evaluation_backtest_mv` — exploded forecast-vs-actual (optional detail)

Pairs predicted vs actual per backtest horizon step. Note: `evaluation_output` stores only the window
*start* date plus arrays (no per-step dates), so this is by horizon position, not calendar date.

In [ ]:
_where_e = f"WHERE e.model = '{MODEL_FILTER}'" if MODEL_FILTER else ""
create_view("evaluation_backtest_mv", f"""
SELECT
  e.unique_id,
  e.model,
  e.backtest_window_start_date,
  z.forecast AS forecast_value,
  z.actual   AS actual_value,
  e.run_id
FROM {src('evaluation_output')} e
LATERAL VIEW explode(arrays_zip(e.forecast, e.actual)) t AS z
{_where_e}
""")

## 6. Validate — row counts, schema, and a sample of each view

In [ ]:
for v in ["scoring_output_mv", "evaluation_metrics_mv", "evaluation_backtest_mv"]:
    fq = f"{CATALOG}.{SCHEMA}.{v}"
    df = spark.table(fq)
    print(f"\n=== {fq} : {df.count():,} rows ===")
    print("  columns:", [f"{f.name}:{f.dataType.simpleString()}" for f in df.schema.fields])
    display(df.limit(5))

In [ ]:
%sql
-- Sanity: Chronos-2 average backtest SMAPE (single model; lower = better).
SELECT model, ROUND(AVG(metric_value), 4) AS avg_smape, COUNT(*) AS n_series
FROM mmf.fresh_retail_net.evaluation_metrics_mv
WHERE metric_name = 'smape'
GROUP BY model;

In [ ]:
%sql
-- Sanity: a single SKU's 7-day forecast, now one row per day.
SELECT unique_id, model, forecast_date, ROUND(forecast_value, 2) AS forecast_value
FROM mmf.fresh_retail_net.scoring_output_mv
ORDER BY unique_id, model, forecast_date
LIMIT 14;

## 7. Next steps

Add these views to the **Fresh Retail Sales Forecasting** Genie space instead of the raw array tables:
- `mmf.fresh_retail_net.daily_sales_raw` (history + covariates)
- `mmf.fresh_retail_net.demand_train` (clean per-SKU history)
- `mmf.fresh_retail_net.scoring_output_mv` (forecasts) ← instead of `scoring_output`
- `mmf.fresh_retail_net.evaluation_metrics_mv` (model accuracy) ← instead of `evaluation_output`

Skip `demand_score` (MMF scaffolding). All views are built only from the original MMF run output.